In [ ]:
%run ./utils_common

In [ ]:
import time
from datetime import date
from decimal import Decimal
from typing import Any, Dict, List, Optional, Tuple

import boto3
from botocore.exceptions import ClientError

In [ ]:
logger = setup_logger('AWSCostExplorer')
logging.getLogger('boto3').setLevel(logging.WARNING)
logging.getLogger('botocore').setLevel(logging.WARNING)

In [ ]:
dbutils.widgets.text('catalog', '', 'CATALOG')
dbutils.widgets.text('schema', '', 'SCHEMA')
dbutils.widgets.text('overlap_days', '3', 'Overlap days (min 2)')

In [ ]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
overlap_days = get_overlap_days(dbutils.widgets.get('overlap_days'), logger=logger)

In [ ]:
# =======================================================
# AWS Cost Client
# =======================================================


class AWSCostClient:
    """Client for querying AWS Cost Explorer API for Databricks cluster costs.

    Uses Databricks service credentials to assume an IAM role with
    ce:GetCostAndUsage permissions. The Cost Explorer API endpoint
    is only available in us-east-1, regardless of where resources run.

    Queries costs grouped by ClusterId tag AND SERVICE dimension,
    filtered to the cluster-attributable EC2 family (compute + EC2-Other,
    which folds in EBS). The caller sums these into a single cloud_cost.
    """

    CE_REGION = 'us-east-1'

    # Only the EC2 family carries the ClusterId tag in this account, so only
    # these two services are cluster-attributable. EBS folds into "EC2 - Other";
    # the dropped S3/ELB/DataTransfer/VPC lines are account-wide shared infra
    # that carry no ClusterId (verified zero tagged cost — see CP10 recon).
    DEFAULT_SERVICES = [
        'Amazon Elastic Compute Cloud - Compute',
        'EC2 - Other',  # EC2 ancillary incl. EBS on cluster instances
    ]

    # Pool EC2 cost (plan §4.2/§4.3, CP5). Idle/warm pool capacity is tagged
    # DatabricksInstancePoolId but NOT ClusterId, so it is invisible to the
    # cluster explorer. We group by BOTH tags and keep only the ClusterId-free
    # slice (the §4.3 netting guard) so pool and cluster cloud cost stay
    # disjoint — no double counting across tabs.
    POOL_TAG_KEY = 'DatabricksInstancePoolId'
    CLUSTER_TAG_KEY = 'ClusterId'

    MAX_CHUNK_DAYS = 30
    MAX_RETRIES = 5
    BASE_RETRY_DELAY = 5

    def __init__(self, service_credential_name: str = 'dbspend-read-ce'):
        session = boto3.Session(
            botocore_session=dbutils.credentials.getServiceCredentialsProvider(
                service_credential_name
            ),
            region_name=self.CE_REGION,
        )
        self.client = session.client('ce')

    # -------- Public API --------

    def get_cluster_costs_daily(
        self,
        start_date: date,
        end_date: date,
        tag_key: str = 'ClusterId',
        services: Optional[List[str]] = None,
        metric: str = 'AmortizedCost',
    ):
        """Query CE for per-cluster daily costs with per-service detail.

        Uses dual GroupBy (TAG + SERVICE dimension) in a single API call.
        Returns per-service rows for the EC2 family that the caller sums
        into a single cloud_cost bucket.

        Returns:
            Spark DataFrame with columns: cluster_id, service_name, cost,
            currency, cost_incurred_date — or None if no cost data found.
        """
        if services is None:
            services = self.DEFAULT_SERVICES

        chunks = self._build_chunks(start_date, end_date)
        all_rows: List[Dict[str, Any]] = []

        for i, (chunk_start, chunk_end) in enumerate(chunks):
            logger.info(f'Querying CE chunk {i+1}/{len(chunks)}: {chunk_start} → {chunk_end}')
            rows = self._query_with_retries(
                chunk_start, chunk_end, tag_key, services, metric
            )
            all_rows.extend(rows)
            if len(chunks) > 1:
                time.sleep(1)

        if not all_rows:
            return None

        return self._rows_to_spark_df(all_rows)

    def get_pool_costs_daily(
        self,
        start_date: date,
        end_date: date,
        services: Optional[List[str]] = None,
        metric: str = 'AmortizedCost',
    ):
        """Query CE for per-pool daily EC2 costs, ClusterId-netted (plan §4.2/§4.3).

        Uses dual GroupBy (TAG DatabricksInstancePoolId + TAG ClusterId) in a
        single API call. The parser keeps ONLY the slice with no ClusterId, so
        the result is disjoint from dbspend360_cloud_cost_explorer (the §4.3
        netting guard) — pool and cluster cloud cost never overlap.

        Returns:
            Spark DataFrame with columns: instance_pool_id, cost, currency,
            cost_incurred_date — or None if no (netted) pool cost found.
        """
        if services is None:
            services = self.DEFAULT_SERVICES

        chunks = self._build_chunks(start_date, end_date)
        all_rows: List[Dict[str, Any]] = []

        for i, (chunk_start, chunk_end) in enumerate(chunks):
            logger.info(
                f'Querying pool CE chunk {i+1}/{len(chunks)}: '
                f'{chunk_start} → {chunk_end}'
            )
            params = self._build_pool_ce_params(
                chunk_start, chunk_end, services, metric
            )
            rows = self._query_params_with_retries(
                params, metric, parser=self._parse_pool_response
            )
            all_rows.extend(rows)
            if len(chunks) > 1:
                time.sleep(1)

        if not all_rows:
            return None

        return self._pool_rows_to_spark_df(all_rows)

    # -------- Chunking --------

    def _build_chunks(self, start: date, end: date) -> List[Tuple[date, date]]:
        chunks = []
        current = start
        while current <= end:
            chunk_end = min(current + timedelta(days=self.MAX_CHUNK_DAYS - 1), end)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- CE params construction --------

    def _build_ce_params(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> dict:
        """Build GetCostAndUsage request parameters.

        Uses dual GroupBy: TAG (ClusterId) + DIMENSION (SERVICE) so we
        get per-service costs per cluster in a single API call.
        CE TimePeriod.End is exclusive, so we add 1 day.
        """
        return {
            'TimePeriod': {
                'Start': start.isoformat(),
                'End': (end + timedelta(days=1)).isoformat(),
            },
            'Granularity': 'DAILY',
            'Metrics': [metric],
            'GroupBy': [
                {'Type': 'TAG', 'Key': tag_key},
                {'Type': 'DIMENSION', 'Key': 'SERVICE'},
            ],
            'Filter': {'Dimensions': {'Key': 'SERVICE', 'Values': services}},
        }

    def _build_pool_ce_params(
        self,
        start: date,
        end: date,
        services: List[str],
        metric: str,
    ) -> dict:
        """Build GetCostAndUsage params for the pool path (plan §4.2/§4.3).

        Dual GroupBy: TAG (DatabricksInstancePoolId) + TAG (ClusterId) so the
        parser can keep only the ClusterId-free slice. The SERVICE filter still
        restricts to the EC2 family even though SERVICE is not a GroupBy here
        (CE allows at most two GroupBy entries). CE TimePeriod.End is exclusive.
        """
        return {
            'TimePeriod': {
                'Start': start.isoformat(),
                'End': (end + timedelta(days=1)).isoformat(),
            },
            'Granularity': 'DAILY',
            'Metrics': [metric],
            'GroupBy': [
                {'Type': 'TAG', 'Key': self.POOL_TAG_KEY},
                {'Type': 'TAG', 'Key': self.CLUSTER_TAG_KEY},
            ],
            'Filter': {'Dimensions': {'Key': 'SERVICE', 'Values': services}},
        }

    # -------- Retry logic --------

    def _query_with_retries(
        self,
        start: date,
        end: date,
        tag_key: str,
        services: List[str],
        metric: str,
    ) -> List[Dict[str, Any]]:
        """Execute a CE query with exponential-backoff retries.

        Handles AWS CE rate limiting (LimitExceededException) with
        progressively longer delays, and retries transient errors.
        """
        params = self._build_ce_params(start, end, tag_key, services, metric)
        return self._query_params_with_retries(params, metric)

    def _query_params_with_retries(
        self, params: dict, metric: str, parser=None
    ) -> List[Dict[str, Any]]:
        """Execute a pre-built CE query with exponential-backoff retries.

        Generic over the response parser so the cluster path
        (``_parse_response``) and the pool path (``_parse_pool_response``)
        share the same rate-limit / transient-error handling.
        """
        last_exception = None

        for attempt in range(self.MAX_RETRIES):
            try:
                return self._execute_paginated_query(params, metric, parser=parser)
            except ClientError as e:
                last_exception = e
                error_code = e.response['Error']['Code']

                if error_code == 'LimitExceededException':
                    wait = min(self.BASE_RETRY_DELAY * (2 ** attempt), 120)
                    logger.warning(
                        f'Rate limited (attempt {attempt + 1}/{self.MAX_RETRIES}), '
                        f'waiting {wait}s'
                    )
                    time.sleep(wait)
                elif attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f'ClientError {error_code} (attempt {attempt + 1}), '
                        f'retrying in {wait}s: {e}'
                    )
                    time.sleep(wait)
                else:
                    raise
            except Exception as e:
                last_exception = e
                if attempt < self.MAX_RETRIES - 1:
                    wait = 2 ** attempt
                    logger.warning(
                        f'Unexpected error (attempt {attempt + 1}), '
                        f'retrying in {wait}s: {e}'
                    )
                    time.sleep(wait)
                else:
                    raise

        raise last_exception

    # -------- Pagination --------

    def _execute_paginated_query(
        self, params: dict, metric: str, parser=None
    ) -> List[Dict[str, Any]]:
        """Execute a CE query, following pagination tokens to completion.

        ``parser`` defaults to the cluster parser; the pool path passes
        ``_parse_pool_response`` so the same pagination loop serves both.
        """
        if parser is None:
            parser = self._parse_response
        rows: List[Dict[str, Any]] = []
        request_params = params.copy()

        while True:
            response = self.client.get_cost_and_usage(**request_params)
            rows.extend(parser(response, metric))

            next_token = response.get('NextPageToken')
            if not next_token:
                break

            request_params['NextPageToken'] = next_token
            time.sleep(0.5)

        return rows

    # -------- Response parsing --------

    def _parse_response(
        self, response: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Parse a single CE response page into flat row dicts.

        With dual GroupBy, each group has two Keys:
          keys[0] = "TagKey$TagValue" (e.g. "ClusterId$0101-abcdef")
          keys[1] = SERVICE dimension value (e.g. "Amazon Elastic Compute Cloud - Compute")
        """
        rows = []
        for time_block in response.get('ResultsByTime', []):
            period_date = time_block['TimePeriod']['Start']

            for group in time_block.get('Groups', []):
                keys = group.get('Keys', [])
                if len(keys) < 2:
                    continue

                raw_tag = keys[0]
                cluster_id = raw_tag.split('$')[-1] if '$' in raw_tag else raw_tag
                service_name = keys[1]

                metric_data = group['Metrics'].get(metric, {})
                amount = float(Decimal(metric_data.get('Amount', '0')))
                currency = metric_data.get('Unit', 'USD')

                if amount == 0.0:
                    continue

                rows.append({
                    'cluster_id': cluster_id,
                    'service_name': service_name,
                    'cost': amount,
                    'currency': currency,
                    'cost_incurred_date': period_date,
                })

        return rows

    def _parse_pool_response(
        self, response: dict, metric: str
    ) -> List[Dict[str, Any]]:
        """Parse a pool CE response page, applying the §4.3 netting guard.

        With dual TAG GroupBy, each group has two Keys:
          keys[0] = "DatabricksInstancePoolId$<pool>"
          keys[1] = "ClusterId$<cluster>" (the value is EMPTY when the pooled
                    instance carries no ClusterId — idle/warm capacity)

        We keep ONLY ClusterId-free rows. The dual-tagged slice (pool VMs that
        also carry a ClusterId) is already captured by the cluster explorer, so
        dropping it here keeps the two tables disjoint and prevents the tiny
        historical overlap (§4.3) from being double-counted.
        """
        rows = []
        for time_block in response.get('ResultsByTime', []):
            period_date = time_block['TimePeriod']['Start']

            for group in time_block.get('Groups', []):
                keys = group.get('Keys', [])
                if len(keys) < 2:
                    continue

                raw_pool = keys[0]
                pool_id = raw_pool.split('$')[-1] if '$' in raw_pool else raw_pool
                raw_cluster = keys[1]
                cluster_id = (
                    raw_cluster.split('$')[-1] if '$' in raw_cluster else raw_cluster
                )

                # Netting guard (§4.3 #2): skip any cost that ALSO carries a
                # ClusterId — it lives in dbspend360_cloud_cost_explorer.
                if cluster_id and cluster_id.strip():
                    continue
                if not pool_id or not pool_id.strip():
                    continue

                metric_data = group['Metrics'].get(metric, {})
                amount = float(Decimal(metric_data.get('Amount', '0')))
                currency = metric_data.get('Unit', 'USD')

                if amount == 0.0:
                    continue

                rows.append({
                    'instance_pool_id': pool_id,
                    'cost': amount,
                    'currency': currency,
                    'cost_incurred_date': period_date,
                })

        return rows

    # -------- Spark conversion --------

    def _rows_to_spark_df(self, rows: List[Dict[str, Any]]):
        """Convert parsed rows to a Spark DataFrame with proper date types."""
        df = spark.createDataFrame(rows)
        return df.withColumn(
            'cost_incurred_date',
            F.to_date(F.col('cost_incurred_date'), 'yyyy-MM-dd'),
        )

    def _pool_rows_to_spark_df(self, rows: List[Dict[str, Any]]):
        """Convert parsed pool rows to a Spark DataFrame with proper date types."""
        df = spark.createDataFrame(rows)
        return df.withColumn(
            'cost_incurred_date',
            F.to_date(F.col('cost_incurred_date'), 'yyyy-MM-dd'),
        )

In [ ]:
# =======================================================
# APP
# =======================================================
class AWSCostReporterApp:
    """Orchestrates incremental AWS cost ingestion into the cloud cost table.

    Reads the audit log to determine the last successful run, queries
    AWS Cost Explorer for the incremental window (with overlap for
    idempotent MERGE), and upserts per-cluster daily cost.

    On AWS, only tagged EC2 + EBS cost is cluster-attributable, so we write a
    single ``cloud_cost`` bucket (sum of the two EC2 services) and leave the
    compute/storage/network/other segments NULL — the segmented split is not
    trustworthy on a shared multi-workload account (see plan §1, §4.1). The
    NULL segments mirror the Azure fallback shape; the frontend gate renders
    the single ``EC2 / EBS`` bucket.
    """

    TABLE_NAME = 'dbspend360_cloud_cost_explorer'
    # Pool EC2 explorer target (plan §4.2, CP5). Separate audit watermark +
    # table so the pool path backfills and MERGEs independently of clusters.
    POOL_TABLE_NAME = 'dbspend360_pool_cloud_cost_explorer'

    # Post-write monitor (plan §4.8b, CP12). Window cloud_cost below this floor
    # ⇒ suspected ClusterId tagging lapse; a non-silent alarm fires.
    CLOUD_COST_FLOOR = 0.01

    # Cutover date when AWS cloud_cost changed definition from "classified
    # multi-service sum" to "tagged EC2+EBS sum" (plan §4.8 MEDIUM-4). Stamped
    # into the audit message so trend charts spanning the deploy aren't misread
    # as a real spend drop — pre/post cloud_cost are NOT directly comparable.
    CUTOVER_DATE = '2026-06-22'

    def __init__(self, catalog, schema, overlap_days, logger):
        self.overlap_days = overlap_days
        self.logger = logger
        self.audit_table = build_table_fqn(catalog, schema, 'dbspend360_audit_log')
        self.target_table = build_table_fqn(catalog, schema, 'dbspend360_cloud_cost_explorer')
        self.pool_target_table = build_table_fqn(catalog, schema, self.POOL_TABLE_NAME)
        # Pool DBU table — read-only, best-effort cross-check for the pool
        # monitor (§4.6): only alarm on ~0 pool cloud when pool DBU exists.
        self.pool_dbu_table = build_table_fqn(catalog, schema, 'dbspend360_pool_dbu_cost')
        self.error_log_table = build_table_fqn(catalog, schema, 'dbspend360_error_log')
        self.client = AWSCostClient()

    def run(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            self.logger.info(
                f'Querying AWS CE cost from {start_dt} to {end_dt} '
                f'(overlap_days={self.overlap_days})'
            )

            valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
            if not valid:
                raise DataQualityError(msg)

            ensure_cost_columns(self.target_table, logger=self.logger)

            spark_df = self.client.get_cluster_costs_daily(
                start_date=start_dt,
                end_date=end_dt,
            )

            quality_msg = f'overlap_days={self.overlap_days}'
            total_cloud_cost = 0.0

            if spark_df is None or spark_df.limit(1).count() == 0:
                self.logger.info('No AWS cost data returned for the requested range.')
                merged_row_count = 0
            else:
                inc_df = filter_valid_cost_rows(spark_df)

                if inc_df.limit(1).count() == 0:
                    self.logger.info('No rows after filtering by cluster_id and cost_incurred_date.')
                    merged_row_count = 0
                else:
                    # Single attributed bucket: sum tagged EC2 + EBS into
                    # cloud_cost, leave segments NULL (mirrors Azure fallback).
                    # No classification, no other-cost breakdown — only tagged
                    # EC2/EBS is cluster-attributable on a shared account.
                    agg_df = safe_cache(
                        inc_df
                        .groupBy('cluster_id', 'currency', 'cost_incurred_date')
                        .agg(F.sum('cost').alias('cloud_cost'))
                        .withColumn('compute_cost', F.lit(None).cast('double'))
                        .withColumn('storage_cost', F.lit(None).cast('double'))
                        .withColumn('network_cost', F.lit(None).cast('double'))
                        .withColumn('other_cost', F.lit(None).cast('double'))
                        .withColumn('created_at', F.current_timestamp())
                        .withColumn('updated_at', F.current_timestamp())
                    )
                    merged_row_count = agg_df.count()
                    # Capture the window total while agg_df is still cached, so
                    # the post-write monitor (§4.8b) need not recompute it.
                    total_cloud_cost = float(
                        agg_df.agg(F.sum('cloud_cost')).collect()[0][0] or 0.0
                    )

                    validate_source_schema(
                        agg_df,
                        {'cluster_id': 'string', 'currency': 'string',
                         'cost_incurred_date': 'date', 'cloud_cost': 'double'},
                        self.target_table, self.logger,
                    )
                    validate_no_negative_costs(
                        agg_df,
                        ['cloud_cost', 'compute_cost', 'storage_cost', 'network_cost', 'other_cost'],
                        self.target_table, self.logger,
                    )
                    validate_currency_consistency(agg_df, 'currency', self.target_table, self.logger)

                    # Single EC2/EBS bucket has no classification coverage.
                    # Deliberately omit the parseable `classification_coverage=`
                    # token so the audit log records no fake 100% coverage for
                    # AWS (get_classification_coverage_trend keys on that token).
                    quality_msg = (
                        f'overlap_days={self.overlap_days}, rows={merged_row_count}, '
                        f'classification=n/a (single-bucket EC2/EBS)'
                    )
                    self.logger.info(f'Data quality: {quality_msg}')

                    merge_cloud_cost_explorer(self.target_table, agg_df)
                    safe_unpersist(agg_df)

                    merge_metrics = get_merge_metrics(self.target_table, self.logger)
                    quality_msg += (
                        f", merge_inserted={merge_metrics.get('num_inserted', '?')}"
                        f", merge_updated={merge_metrics.get('num_updated', '?')}"
                    )

                    validate_post_merge(
                        self.target_table, 'cost_incurred_date',
                        start_dt, end_dt, merged_row_count, self.logger,
                    )

            # Post-write monitor / alarm (plan §4.8b, CP12). Runs on every
            # path — including the no-data path, where a ~0 cloud_cost is the
            # tagging-lapse signal. Advisory: never fails the run.
            monitor_alerts = self._monitor_post_write(
                spark_df, total_cloud_cost, start_dt, end_dt,
            )

            # Stamp the cutover date (MEDIUM-4 trend comparability) and the
            # monitor result into the audit message on every path.
            quality_msg += (
                f', cloud_cost_def=tagged_ec2_ebs(cutover={self.CUTOVER_DATE})'
                f', monitor_alerts={len(monitor_alerts)}'
            )

            self.logger.info(
                f'Merged {merged_row_count} rows into {self.target_table} '
                f'for {start_dt} → {end_dt} (overlap_days={self.overlap_days}).'
            )

            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                'SUCCESS', merged_row_count, quality_msg,
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f'Run failed: {msg}')
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, 'FAILED', 0, msg,
                )
            except Exception:
                self.logger.error('Failed to write FAILED audit entry')
            raise

    def run_pool(self):
        """Pool-tag EC2 explorer (plan §4.2/§4.3/§4.6, CP5).

        Any failure logs to error_log, writes a FAILED pool audit row, and
        propagates so downstream pool rollups cannot refresh against stale
        cloud data while the job reports SUCCESS. The path keeps its own audit
        watermark (POOL_TABLE_NAME), so it still backfills independently.
        """
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(
                self.audit_table, self.POOL_TABLE_NAME, self.overlap_days
            )

            self.logger.info(
                f'Querying AWS CE POOL cost from {start_dt} to {end_dt} '
                f'(overlap_days={self.overlap_days})'
            )

            valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
            if not valid:
                raise DataQualityError(msg)

            spark_df = self.client.get_pool_costs_daily(
                start_date=start_dt,
                end_date=end_dt,
            )

            quality_msg = f'overlap_days={self.overlap_days}'
            total_pool_cost = 0.0

            if spark_df is None or spark_df.limit(1).count() == 0:
                self.logger.info('No AWS pool cost data returned for the requested range.')
                merged_row_count = 0
            else:
                inc_df = (
                    spark_df
                    .filter(
                        (F.col('instance_pool_id').isNotNull())
                        & (F.col('instance_pool_id') != '')
                    )
                    .filter(F.col('cost_incurred_date').isNotNull())
                )

                if inc_df.limit(1).count() == 0:
                    self.logger.info('No pool rows after filtering by pool_id and date.')
                    merged_row_count = 0
                else:
                    # Single attributed bucket per (pool, day, currency); the
                    # ClusterId netting already happened in the parser (§4.3).
                    # Segments + idle/active stay NULL (reserved — §4.5).
                    agg_df = safe_cache(
                        inc_df
                        .groupBy('instance_pool_id', 'currency', 'cost_incurred_date')
                        .agg(F.sum('cost').alias('cloud_cost'))
                        .withColumn('compute_cost', F.lit(None).cast('double'))
                        .withColumn('storage_cost', F.lit(None).cast('double'))
                        .withColumn('network_cost', F.lit(None).cast('double'))
                        .withColumn('other_cost', F.lit(None).cast('double'))
                        .withColumn('created_at', F.current_timestamp())
                        .withColumn('updated_at', F.current_timestamp())
                    )
                    merged_row_count = agg_df.count()
                    total_pool_cost = float(
                        agg_df.agg(F.sum('cloud_cost')).collect()[0][0] or 0.0
                    )

                    validate_source_schema(
                        agg_df,
                        {'instance_pool_id': 'string', 'currency': 'string',
                         'cost_incurred_date': 'date', 'cloud_cost': 'double'},
                        self.pool_target_table, self.logger,
                    )
                    validate_no_negative_costs(
                        agg_df, ['cloud_cost'], self.pool_target_table, self.logger,
                    )
                    validate_currency_consistency(
                        agg_df, 'currency', self.pool_target_table, self.logger,
                    )

                    quality_msg = (
                        f'overlap_days={self.overlap_days}, rows={merged_row_count}, '
                        f'classification=n/a (single-bucket EC2/EBS), '
                        f'netting=ClusterId-excluded'
                    )
                    self.logger.info(f'Pool data quality: {quality_msg}')

                    merge_pool_cloud_cost_explorer(self.pool_target_table, agg_df)
                    safe_unpersist(agg_df)

                    merge_metrics = get_merge_metrics(self.pool_target_table, self.logger)
                    quality_msg += (
                        f", merge_inserted={merge_metrics.get('num_inserted', '?')}"
                        f", merge_updated={merge_metrics.get('num_updated', '?')}"
                    )

                    validate_post_merge(
                        self.pool_target_table, 'cost_incurred_date',
                        start_dt, end_dt, merged_row_count, self.logger,
                    )

            monitor_alerts = self._monitor_pool_post_write(
                total_pool_cost, start_dt, end_dt,
            )

            quality_msg += (
                f', cloud_cost_def=tagged_ec2_ebs_pool(cutover={self.CUTOVER_DATE})'
                f', monitor_alerts={len(monitor_alerts)}'
            )

            self.logger.info(
                f'Merged {merged_row_count} pool rows into {self.pool_target_table} '
                f'for {start_dt} → {end_dt} (overlap_days={self.overlap_days}).'
            )

            log_audit_run(
                self.audit_table, self.POOL_TABLE_NAME, start_dt, end_dt,
                'SUCCESS', merged_row_count, quality_msg,
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f'Pool explorer run failed: {msg}')
            try:
                write_error_log_entries(
                    [f'Pool CE explorer failed for {start_dt} → {end_dt}: {msg}'],
                    'AWS', 'POOL_COST_EXPLORER_FAILED', self.error_log_table,
                )
            except Exception:
                self.logger.error('Failed to persist pool explorer failure to error_log')
            try:
                log_audit_run(
                    self.audit_table, self.POOL_TABLE_NAME, start_dt, end_dt,
                    'FAILED', 0, msg,
                )
            except Exception:
                self.logger.error('Failed to write FAILED pool audit entry')
            # Pool cloud is part of the tab's headline total. Propagate failure
            # so downstream rollups cannot refresh from a stale explorer while
            # the job reports SUCCESS.
            raise

    def _pool_dbu_present(self, start_dt, end_dt):
        """Best-effort check: does pool DBU exist in the window? (§4.6 monitor).

        Used only to suppress a false ~0-cloud alarm on a genuinely idle/empty
        window. Conservative: returns False (no alarm) if the table can't be
        read, so the monitor never fails the run.
        """
        try:
            cnt = (
                spark.table(self.pool_dbu_table)
                .filter(
                    (F.col('usage_date') >= F.lit(start_dt))
                    & (F.col('usage_date') <= F.lit(end_dt))
                )
                .limit(1)
                .count()
            )
            return cnt > 0
        except Exception as e:
            self.logger.warning(
                f'Could not read pool DBU table for monitor cross-check: {e}'
            )
            return False

    def _monitor_pool_post_write(self, total_pool_cost, start_dt, end_dt):
        """Post-write monitor / alarm for the pool path (plan §4.6).

        If window pool cloud_cost collapses to ~0 while pool DBU is non-zero,
        raise a non-silent alarm (suspected DatabricksInstancePoolId tag lapse
        or empty Cost Explorer response). A ~0 cloud with no pool DBU is a
        genuinely idle/empty window, not an error, so no alarm fires there.
        Advisory only — never fails the run. Returns the alert list.
        """
        alerts = []

        if total_pool_cost < self.CLOUD_COST_FLOOR:
            if self._pool_dbu_present(start_dt, end_dt):
                alerts.append(
                    f'AWS pool cloud_cost for {start_dt} → {end_dt} collapsed to '
                    f'{total_pool_cost:.4f} (< {self.CLOUD_COST_FLOOR}) while pool '
                    f'DBU is non-zero; suspected DatabricksInstancePoolId tag '
                    f'lapse or empty Cost Explorer response.'
                )
            else:
                self.logger.info(
                    f'Pool cloud_cost ~0 for {start_dt} → {end_dt} but no pool DBU '
                    f'in window; treating as genuinely idle/empty — no alarm.'
                )

        if alerts:
            for alert in alerts:
                self.logger.error(f'[POOL MONITOR ALARM] {alert}')
            try:
                write_error_log_entries(
                    alerts, 'AWS', 'POOL_COST_MONITOR_ALARM', self.error_log_table,
                )
            except Exception as e:
                self.logger.error(
                    f'Failed to persist pool monitor alarm to error_log: {e}'
                )
        else:
            self.logger.info(
                f'Pool post-write monitor OK for {start_dt} → {end_dt}: '
                f'cloud_cost={total_pool_cost:.4f}.'
            )

        return alerts

    def _monitor_post_write(self, spark_df, total_cloud_cost, start_dt, end_dt):
        """Post-write monitor / alarm (plan §4.8b, CP12).

        Non-silent guard against the two known regression modes of the new
        tagged-EC2 ``cloud_cost`` definition. AWS no longer has any other
        cross-check (the coverage badge and other-cost breakdown were removed
        in CP11), so a silent regression here would shrink attributed cluster
        spend with nothing to catch it:

          1. ``cloud_cost`` for the window collapses to ~0 → suspected
             ``ClusterId`` tagging lapse (or an empty Cost Explorer response).
          2. one or both expected EC2 services are absent from the CE response
             → suspected AWS service rename / filter drift; the 2-service
             SERVICE filter returns one or zero of its expected services and
             may be silently dropping cluster-attributable cost.

        Each tripped mode logs at ERROR and appends an error_log row so the
        signal is not silent. Advisory only — never fails the run. Returns the
        list of alert strings (empty == healthy).
        """
        alerts = []

        if total_cloud_cost < self.CLOUD_COST_FLOOR:
            alerts.append(
                f'AWS cloud_cost for {start_dt} → {end_dt} collapsed to '
                f'{total_cloud_cost:.4f} (< {self.CLOUD_COST_FLOOR}); suspected '
                f'ClusterId tagging lapse or empty Cost Explorer response.'
            )

        seen_services = set()
        if spark_df is not None:
            seen_services = {
                row['service_name']
                for row in spark_df.select('service_name').distinct().collect()
            }
        missing_services = [
            svc for svc in self.client.DEFAULT_SERVICES if svc not in seen_services
        ]
        if missing_services:
            alerts.append(
                f'AWS CE response missing expected EC2 service(s): '
                f'{missing_services} (seen={sorted(seen_services)}); suspected '
                f'AWS service rename — the SERVICE filter may be silently '
                f'dropping cluster-attributable cost.'
            )

        if alerts:
            for alert in alerts:
                self.logger.error(f'[MONITOR ALARM] {alert}')
            try:
                write_error_log_entries(
                    alerts, 'AWS', 'COST_MONITOR_ALARM', self.error_log_table,
                )
            except Exception as e:
                self.logger.error(
                    f'Failed to persist monitor alarm to error_log: {e}'
                )
        else:
            self.logger.info(
                f'Post-write monitor OK for {start_dt} → {end_dt}: '
                f'cloud_cost={total_cloud_cost:.4f}, '
                f'services={sorted(seen_services)}.'
            )

        return alerts

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AWSCostReporterApp(catalog, schema, overlap_days, logger)
# Both paths are required for a successful task. Pool-cloud failures propagate
# so downstream pool rollups cannot consume stale data.
app.run()
app.run_pool()